In [1]:
import psycopg2
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from dotenv import load_dotenv
import os

d:\python-workspace\NL2SQL-chat\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [2]:
# Khởi tạo model và adapter
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
compute_dtype = torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    dtype=compute_dtype,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")

adapter_path = r"..\save_model\checkpoint-39290"
new_model = PeftModel.from_pretrained(model, adapter_path)
print("Tải mô hình thành công !")

Loading weights: 100%|██████████| 434/434 [00:18<00:00, 22.94it/s]


Tải mô hình thành công !


In [3]:
model.eval()
def generate_response(schema_db, user_query, wrong_query=""):
    if wrong_query: wrong_query = f"by modifying this query: '{wrong_query}'"
    message = [
        {"role":"system", "content": f"""You are a highly specialized NL2SQL engine. Your ONLY task is to translate natural language into accurate SQL queries based on the provided schema.

CRITICAL RULES:
1. Output EXACTLY ONE valid SQL query.
2. DO NOT include any greetings, explanations, or conversational text before or after the query.
3. DO NOT wrap the SQL query in markdown formatting blocks (e.g., strictly NO ```sql or ``` tags). Return raw text only.

Schema: \n{schema_db}"""},
        {"role":"user", "content": f"{user_query} {wrong_query}"}
    ]
    input_text = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )
    tokens = tokenizer(input_text, return_tensors="pt").to(device)
    output = new_model.generate(
        **tokens,
        max_new_tokens=200,
        pad_token_id=tokenizer.eos_token_id
    )
    output_decode = tokenizer.decode(output[0])
    return output_decode

schema_db = """-- 1. Bảng Users
CREATE TABLE Users (
    user_id SERIAL PRIMARY KEY,
    full_name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    password_hash VARCHAR(255) NOT NULL,
    phone VARCHAR(20),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- 2. Bảng Categories
CREATE TABLE Categories (
    category_id SERIAL PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL,
    description TEXT
);

-- 3. Bảng Products 
CREATE TABLE Products (
    product_id SERIAL PRIMARY KEY,
    product_name VARCHAR(255) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    stock_quantity INT NOT NULL DEFAULT 0,
    description TEXT,
    category_id INT,
    FOREIGN KEY (category_id) REFERENCES Categories(category_id) ON DELETE SET NULL
);

-- 4. Bảng Shopping_Cart 
CREATE TABLE Shopping_Cart (
    cart_id SERIAL PRIMARY KEY,
    user_id INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE
);

-- 5. Bảng Cart_Items 
CREATE TABLE Cart_Items (
    cart_item_id SERIAL PRIMARY KEY,
    cart_id INT,
    product_id INT,
    quantity INT NOT NULL DEFAULT 1,
    FOREIGN KEY (cart_id) REFERENCES Shopping_Cart(cart_id) ON DELETE CASCADE,
    FOREIGN KEY (product_id) REFERENCES Products(product_id) ON DELETE CASCADE
);

-- 6. Bảng Orders 
CREATE TABLE Orders (
    order_id SERIAL PRIMARY KEY,
    user_id INT,
    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    total_amount DECIMAL(12, 2) NOT NULL,
    status VARCHAR(50) DEFAULT 'Pending', 
    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE SET NULL
);

-- 7. Bảng Order_Items 
CREATE TABLE Order_Items (
    order_item_id SERIAL PRIMARY KEY,
    order_id INT,
    product_id INT,
    quantity INT NOT NULL,
    price_at_purchase DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id) ON DELETE CASCADE,
    FOREIGN KEY (product_id) REFERENCES Products(product_id) ON DELETE SET NULL
);

-- 8. Bảng Payments 
CREATE TABLE Payments (
    payment_id SERIAL PRIMARY KEY,
    order_id INT,
    payment_method VARCHAR(50) NOT NULL, 
    amount DECIMAL(12, 2) NOT NULL,
    status VARCHAR(50) DEFAULT 'Pending', 
    transaction_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id) ON DELETE CASCADE
);

-- 9. Bảng Shipping 
CREATE TABLE Shipping (
    shipping_id SERIAL PRIMARY KEY,
    order_id INT,
    shipping_address TEXT NOT NULL,
    tracking_number VARCHAR(100),
    status VARCHAR(50) DEFAULT 'Preparing', 
    shipped_date TIMESTAMP NULL,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id) ON DELETE CASCADE
);

-- 10. Bảng Reviews 
CREATE TABLE Reviews (
    review_id SERIAL PRIMARY KEY,
    user_id INT,
    product_id INT,
    rating INT CHECK (rating >= 1 AND rating <= 5),
    comment TEXT,
    review_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE SET NULL,
    FOREIGN KEY (product_id) REFERENCES Products(product_id) ON DELETE CASCADE
);"""

In [3]:
DB_NAME = os.getenv("DB_NAME")
USER = os.getenv("USER")
PASS = os.getenv("PASSWORD")

# Khởi tọa kết nối với database
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database=DB_NAME,
    user=USER,
    password=PASS
)

cursor = conn.cursor()
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Đã kết nối thành công! Phiên bản: {db_version[0]}\n")
print("-" * 50)

Đã kết nối thành công! Phiên bản: PostgreSQL 16.9, compiled by Visual C++ build 1944, 64-bit

--------------------------------------------------


In [5]:
num_wait = 3
count_wait = 0
prompt = input("Input: ")
sql_query = ""
while count_wait < num_wait:
    try:
        print("Đang sinh lệnh SQL")
        response = generate_response(schema_db, prompt)
        sql_query = response.split("assistant")[-1][:-10].strip()
        print(f"SQL query: {sql_query}")
        cursor.execute(sql_query)
        results = cursor.fetchall()
        for result in results:
            print(result)
    except:
        count_wait += 1
        print(f"Lệnh SQL không hoạt động, thực hiện sửa lại...\nSố lần chờ: {count_wait}/3\n")
    else:
        count_wait = 0
        break
if count_wait != 0:
    print(f"Model không thể xử lý yêu cầu truy vấn này, in ra lệnh SQL cuối cùng được tạo: {sql_query}")

Đang sinh lệnh SQL
SQL query: SELECT u.full_name, u.email, u.phone FROM Users u JOIN Shopping_Cart sc ON u.user_id = sc.user_id JOIN Cart_Items ci ON sc.cart_id = ci.cart_id WHERE ci.product_id IN (SELECT product_id FROM Products)
('Nguyễn Văn A', 'nguyenvana@email.com', '0901234567')
('Nguyễn Văn A', 'nguyenvana@email.com', '0901234567')
('Trần Thị B', 'tranthib@email.com', '0912345678')
('Phạm Thị D', 'phamthid@email.com', '0922334455')
('Phạm Thị D', 'phamthid@email.com', '0922334455')
('Hoàng Văn E', 'hoangvane@email.com', '0933445566')
('Đặng Thị F', 'dangthif@email.com', '0944556677')


In [6]:
# cursor.close()
# conn.close()